In [2]:
import pandas as pd

# Ladda in CSV dataset
netflix = pd.read_csv("netflix Dataset.csv")

netflix.head()

print(netflix.dtypes)

Show_Id         object
Category        object
Title           object
Director        object
Cast            object
Country         object
Release_Date    object
Rating          object
Duration        object
Type            object
Description     object
dtype: object


In [6]:
netflix['Release_Year'] = pd.to_datetime(netflix['Release_Date'], errors='coerce').dt.year
netflix['Release_Year'] = netflix['Release_Year'].fillna(0).astype(int)
def parse_duration(x):
    if pd.isna(x):
        return 0
    if "Season" in x:   # TV Show
        return int(x.split()[0])   # antal säsonger
    if "min" in x:      # Movie
        return int(x.split()[0])   # antal minuter
    return 0

netflix['Duration_num'] = netflix['Duration'].apply(parse_duration)

print(netflix[['Title', 'Duration', 'Duration_num', 'Release_Date', 'Release_Year']].head(10))
print(netflix.dtypes)

   Title   Duration  Duration_num       Release_Date  Release_Year
0     3%  4 Seasons             4    August 14, 2020          2020
1  07:19     93 min            93  December 23, 2016          2016
2  23:59     78 min            78  December 20, 2018          2018
3      9     80 min            80  November 16, 2017          2017
4     21    123 min           123    January 1, 2020          2020
5     46   1 Season             1       July 1, 2017          2017
6    122     95 min            95       June 1, 2020          2020
7    187    119 min           119   November 1, 2019          2019
8    706    118 min           118      April 1, 2019          2019
9   1920    143 min           143  December 15, 2017          2017
Show_Id         object
Category        object
Title           object
Director        object
Cast            object
Country         object
Release_Date    object
Rating          object
Duration        object
Type            object
Description     object
Release_Ye

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Skapa en features-matris med bara år + duration
X = netflix[['Release_Year', 'Duration_num']]

# Skala för bättre jämförelse
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Exempel: cosine similarity på de två numeriska features
similarity = cosine_similarity(X_scaled)

# Funktion för att hämta de 5 närmsta filmerna
indices = pd.Series(netflix.index, index=netflix['Title']).drop_duplicates()

def get_recommendations(title, n=5):
    if title not in indices:
        return ["Filmen finns ej i datasetet"]
    idx = indices[title]
    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # Skippa sig själv
    movie_indices = [i[0] for i in sim_scores]
    return netflix['Title'].iloc[movie_indices].tolist()

print(get_recommendations("23:59"))


['Balto', 'Balto 3: Wings of Change', 'Expedition China', 'Fishtronaut: The Movie', 'Happy Anniversary']
